# Portfolio Optimization for SFO, paper 2: the non-parametric pipeline

Paper 1 ([`01_analytical_normal_market.ipynb`](01_analytical_normal_market.ipynb)) solved Meucci's Section 6.1 allocation problem in closed form — normal market, CARA utility, certainty-equivalent, VaR/CVaR budget — and validated each numerical replacement against the closed form at a one-week horizon. The derivations live there: the CARA certainty-equivalent and its dollar units, the weights formulation (nondimensionalization), the Rockafellar–Uryasev representation of CVaR, the in-sample optimism bias and the "one judge" comparison protocol.

This paper builds the target pipeline of the architecture diagram below, top to bottom. What changes:

- **No estimation step.** No $\hat\mu$, no $\hat\Sigma$, no normality: the empirical distribution of the weekly invariants *is* the model.
- **Horizon $\tau$ = 1 year** ($k = 52$ weeks): the invariants are projected by a stationary block bootstrap of whole historical weeks.
- **Money market** joins the market distribution as a riskless column.
- **Scenario probabilities $p$** are explicit (Fully Flexible Probabilities): an entropy-pooling stage sits between the market distribution and the optimizer, and every downstream stage — optimizer, CE, CVaR, evaluator, liquidity — is probability-weighted.
- **Liquidity shapes the allocation.** Meucci's (2012) illiquidity model — the expected cost and the uncertainty of what a funding need forces you to sell at the horizon — enters the optimizer as well as the evaluator, with one parameter set and one liquidation policy.
- **Evaluator** after the optimizer: unseen scenarios, history resampling (estimation uncertainty), and Meucci's (2012) liquidity-adjusted P&L.

Fixed throughout: initial wealth $w$ = USD 100M, risk tolerance $\zeta$ = USD 10M, confidence level $c$ = 95%, horizon $\tau$ = 1 year. The CVaR budget is a new, documented placeholder (paper 1's 4.1% was calibrated for one week without cash).

## Architecture

The target pipeline (Meucci's checklist, non-parametric throughout: the empirical distribution of the invariants *is* the model — no estimation step). Dashed-in boxes on the whiteboard that are still open are marked *open*.

```mermaid
flowchart LR
    prices(["Internet prices<br/>weekly, Wednesday close"])

    subgraph INV["Invariants (i.i.d.)"]
        eq["Equity<br/>log returns"]
        re["Real estate<br/><i>open: invariant TBD</i>"]
        bd["Bonds<br/><i>open: invariant TBD</i>"]
    end

    prices --> eq & re & bd
    eq & re & bd --> proj["Joint projection<br/>whole-week bootstrap to the horizon"]

    proj --> eqp["Invariant → price<br/>equity"]
    proj --> rep["Invariant → price<br/>real estate"]
    proj --> bdp["Invariant → price<br/>bonds"]

    mm["Money market"] --> mmproj["Projection<br/>constant at today's yield"]
    rf(["Risk-free rate"]) --> mmproj

    eqp & rep & bdp & mmproj --> md["Market distribution X<br/>non-parametric scenarios"]

    md --> ep["Entropy pooling"]
    p0(["Prior probabilities p₀<br/>uniform"]) --> ep
    views(["Views"]) --> conv["Linear constraints<br/>converter"] --> ep

    ep -->|"X, p"| opt["Portfolio optimization<br/>max CE s.t. CVaR budget,<br/>net of liquidity cost;<br/>funding need met by sales"]

    rt(["Risk tolerance"]) --> opt
    goal(["Goal: absolute portfolio value"]) --> opt
    util(["Utility function: CARA"]) --> opt
    sat(["Index of satisfaction<br/>CE, CVaR"]) --> opt
    cons(["Portfolio constraints<br/>long-only, caps"]) --> opt

    opt -->|weights| evl["Evaluator<br/>CE, CVaR, value on unseen scenarios;<br/>conditional convolution, liquidity score"]
    illiq(["Illiquidity model<br/>spread, impact, funding threshold,<br/>cost-minimizing liquidation policy"])
    illiq -->|"expected cost, cost std,<br/>funding need"| opt
    illiq -->|"conditional convolution"| evl
```

Design rules behind the diagram:

- **One joint projection.** All invariants sit in one matrix on a common weekly grid and are projected together (whole weeks resampled), so cross-asset dependence — including joint crashes — is kept. Projecting asset classes separately and gluing them afterwards would lose it (the copula of 52-week sums is not the weekly copula).
- **Cash joins the market distribution before entropy pooling**, so views stated relative to cash can be expressed. A constant column is untouched by the reweighting.
- **Entropy pooling outputs probabilities `p`, not new scenarios.** Everything downstream (CE, CVaR, liquidity) must be probability-weighted.
- **Liquidity is optimized and evaluated, with one model.** Meucci's (2012) illiquidity model feeds both boxes with the same parameters, the same dollar funding threshold and the same (cost-minimizing) liquidation policy. In the optimizer each scenario's funding need is a recourse decision — the sales are variables next to the weights — and their expected cost (6) and its variance (7)–(9) are charged to that scenario's wealth; the evaluator then grades the chosen weights exactly by conditional convolution. What is sold in a scenario depends on what is held, so liquidity cannot be precomputed into the market distribution: it has to sit inside the optimizer, not in front of it.
- **Two kinds of linear constraints.** View constraints act on probabilities (into entropy pooling); portfolio constraints act on weights (into the optimizer).
- **The evaluator re-scores the chosen weights** on scenarios the optimizer did not see: in-sample CE/CVaR are optimistically biased (the optimizer's curse). Meucci's conditional convolution and liquidity score belong here too.

Open items from the whiteboard: **transaction costs**, **rebalancing** (the one-period model assumes buy-and-hold), **timeline**.


## Scope: where this paper sits on paper 1's roadmap

Paper 1's "practical model" roadmap, item by item, as it stands after this paper:

- **Projection — 1 Year horizon**: done here (block bootstrap to $k = 52$ weeks).
- **Views / beliefs — entropy pooling** ("Fully Flexible Views", 2008: keep the simulated scenarios, reweight their probabilities to satisfy the views with minimal distortion): the *plumbing* is in place — prior $p_0$, an entropy-pooling stage, probability-weighted consumers — but the stage is a pass-through until the solver and the views are added.
- **Estimation — shrinkage, robust M-estimators**: not applicable here; there is no estimation step. Their non-parametric counterparts are conditioning the history (time/state weights on the weeks, already supported by the bootstrap's `probabilities` argument) and views.
- **Invariants — bonds, real estate in Abu Dhabi**: open (the dashed boxes of the diagram); each needs its own invariant before it can join the joint projection.
- **Projection — cyclostationary model**: open.
- **Utility — HARA family**: open; the objective is still the CARA certainty-equivalent.
- **Optimization — transaction costs** (6.14)/(6.23): open (and on the whiteboard's open list below). The cost of *liquidating at the horizon* (Meucci 2012) is now inside the optimizer; the cost of building the allocation today is not.

## Known limitations of the current version

Deliberate simplifications — start simple, refine one block at a time.

- **Money market modeled as a constant.** Cash is a constant column at today's yield (`deposit_rate`, compounded to the horizon). A money market fund's yield floats with the short rate, so beyond a few months its horizon value is random and co-moves with bonds: horizon value $= \prod_t (1 + r_t\,\Delta t)$ over an unknown rate path.
  *Improvement:* add the weekly change in the short rate $\Delta r_t$ (e.g. 3-month T-bill on the Wednesday grid) to the joint invariant matrix, rebuild the path from today's level $r_t = r_{\text{today}} + \sum_{s \le t} \Delta r_s$ (floored near 0), and price cash as $\approx \sum_t r_t\,\Delta t$.
  *Not* the fund's weekly return as the invariant: rate *levels* are persistent, so bootstrapping them would mix the ~0% (2021) and ~5% regimes and earn the sample-average rate instead of starting from today's.


In [1]:
%load_ext autoreload
%autoreload 2

import contextlib
import dataclasses
import io
from datetime import date

import numpy as np
import pandas as pd
from plotly import graph_objects as go
from plotly.subplots import make_subplots

from emfin_capstone import data_types as dt
from emfin_capstone.toolbox import liquidity, price, returns, scenario_market

## Internet prices → invariants (Meucci Ch. 3.1)

Equity invariants are the weekly compounded (log) returns of dividend-adjusted closes, Wednesday-to-Wednesday — constructed exactly as in paper 1 (see its "Invariants: raw data" section for the choices: total return, Wednesday grid, 7-calendar-day rule). Only equities exist today; real estate and bonds are the open boxes of the diagram.

All invariants sit on one common weekly grid, and weeks missing for any asset are dropped: the joint projection resamples whole rows.

In [2]:
asset_symbol_list = ['DFUS', 'DFAI', 'DFAE']
start_date = date(year=2021, month=6, day=15)
end_date = date.today()
price_fetcher_factory = price.PriceFetcherFactory(adjust_prices=True)
price_fetcher = price_fetcher_factory.build_price_fetcher(
    asset_type=dt.AssetType.EQUITY,
    exchange=dt.Exchange.NYSE,
)

prices_dict = {}
weekly_returns_dict = {}
for asset_symbol in asset_symbol_list:
    price_history = price_fetcher.fetch_symbol(symbol=asset_symbol, start=start_date, end=end_date)
    prices_dict[asset_symbol] = pd.Series(
        {observation.date: observation.close for observation in price_history.closes}
    )
    weekly_returns = returns.convert_history_prices_to_weekly_returns(price_history)
    weekly_returns_dict[weekly_returns.symbol] = pd.Series(
        {r.date: r.equity_return for r in weekly_returns.equity_returns}
    )

# Align each column on its own dates: pandas outer-joins the per-symbol indices,
# so assets with differing trading calendars line up automatically.  Weeks missing
# for any asset are dropped from the returns: the bootstrap needs whole rows.
assets_prices = pd.DataFrame(prices_dict)
assets_returns = pd.DataFrame(weekly_returns_dict).dropna()
weekly_log_returns = assets_returns.to_numpy()
prices_vector = assets_prices.iloc[-1].to_numpy().reshape((-1, 1))  # today's prices p_T
print(f"{len(weekly_log_returns)} weeks, {assets_returns.index[0]} to {assets_returns.index[-1]}")
print(assets_returns.head())

276 weeks, 2021-06-23 to 2026-11-04
                DFUS      DFAI      DFAE
2021-06-23  0.005433 -0.020999  0.001914
2021-06-30  0.013287 -0.002051  0.011031
2021-07-07  0.011918  0.003723 -0.018912
2021-07-14  0.000084  0.004117 -0.001377
2021-07-21  0.000105 -0.012643 -0.013875


# Scenario Engine

From this point on, the market distribution enters the optimization only as a
**matrix of sampled scenarios** (with uniform probabilities, so far implicit).
Three things make this section more than a re-implementation of the closed-form
solution:

1. **Estimation is fully decoupled from optimization.** The optimizer consumes
   scenarios and never learns where they came from. Today they are bootstrapped
   historical weeks (stationary block bootstrap, Politis & Romano 1994: whole
   weeks, so cross-asset dependence is kept as observed), with the Monte Carlo
   draws from the lognormal market implied by normal invariants kept as the
   benchmark; tomorrow the *same, unchanged* optimization code accepts Student-t
   invariants, entropy-pooling-reweighted scenario sets, or handcrafted
   stress cases — genuinely any distribution model that can produce samples.
   This makes the model very flexible: every upstream refinement in the
   roadmap (estimation, projection, views) plugs in without touching the
   optimizer.

2. **Constrained optimization on an empirical distribution — via
   Rockafellar–Uryasev.** On scenarios, quantile-based risk measures (VaR) are
   non-convex step functions of the allocation. The R–U minimization formula
   represents CVaR through an auxiliary threshold variable and positive parts,
   keeping the problem convex — the single result that makes the whole
   *constrained* optimization step possible on empirical distributions
   (references in `scenario_market`).

3. **The utility function is still hardwired.** The objective is the empirical
   CARA certainty-equivalent (`log_sum_exp`), so exponential utility remains
   baked into the convex formulation. Generalizing the objective to a pluggable
   utility family (HARA) is the next task in building the scenario engine; the
   requirement on any candidate utility is concavity plus DCP-expressibility.

## Joint projection: whole-week block bootstrap to the horizon (Meucci Ch. 3.2)

Log returns add across time, so a horizon of $k = 52$ weeks is the sum of 52 weekly invariants. Without a parametric model there is no $\tau/\tilde\tau$ scaling of moments; instead each scenario path is built from historical weeks: a stationary block bootstrap (Politis & Romano 1994) draws runs of consecutive weeks with geometric lengths of mean $L$ and sums them, then prices by $P_{T+\tau} = p_T \circ e^{X_\tau}$ — the "invariant → price" box for equities. Whole rows are drawn, so the cross-asset dependence (including joint crashes) is kept as observed, with no copula and no covariance model.

Block length: rule of thumb $L = n^{1/3}$ weeks on $n$ observations. The choice and its side effects are discussed in the note below.

In [3]:
N_SCENARIOS = 400000
SEED = 42
HORIZON_WEEKS = 52  # tau = 1 year

# rule of thumb n^(1/3) (6-7 weeks on ~270): weekly r shows no autocorrelation, r^2 does
# (Politis-White gives ~2-3 on r, ~11-14 on r^2 for DFUS/DFAI, and is unstable).
mean_block_length = round(len(weekly_log_returns) ** (1 / 3))
print(f"mean block length: {mean_block_length} weeks")

mean block length: 7 weeks


> **Note — block vs i.i.d. bootstrap vs filtered historical simulation (not the current focus).**
>
> Weekly returns in the sample are slightly *negatively* autocorrelated (lag-1 ≈ −0.06 to −0.12, not statistically significant — Ljung–Box p ≈ 0.7–0.8), while squared returns are strongly autocorrelated (volatility clustering, mostly the 2022 regime).
> The block bootstrap copies runs of consecutive weeks, so it reproduces *both*: the real clustering and the probably-spurious sign pattern. Over 52 weeks the alternating ups and downs partly cancel — equal-weight annual vol 14.0% (mean block 6 weeks) vs 16.5% (i.i.d. bootstrap), i.e. ~26% less annual variance, driven entirely by the insignificant autocorrelations. The i.i.d. bootstrap avoids that but loses the clustering.
>
> **Filtered historical simulation** keeps the real part and drops the fake one: standardize each week by an EWMA volatility estimate, resample the standardized shocks i.i.d., and rebuild each path by re-applying the volatility recursion from today's level. Still non-parametric (the shocks are the data); one modelling choice, the EWMA decay. *(Numbers from the 274-week analysis of 2026-10-27.)*


## Money market: constant at today's yield

The money market is projected outside the bootstrap — a constant horizon value at today's yield, $1\cdot(1 + r)^{\tau}$ — and joins the market distribution as one more column **before** entropy pooling, so that views can be stated relative to cash; a constant column is untouched by any reweighting. Why a constant and what it misses: see *Known limitations* above.

In [4]:
cash_price = 1.0
deposit_rate = 0.03  # today's money-market yield, annual
tau_years = HORIZON_WEEKS / 52
cash_horizon_value = cash_price * (1 + deposit_rate) ** tau_years
prices_with_cash = np.append(prices_vector.ravel(), cash_price)
asset_names = asset_symbol_list + ['Cash']
print(f"cash horizon value: {cash_horizon_value:.4f}")

cash horizon value: 1.0300


## Market distribution $X$: non-parametric scenarios

The market distribution is the scenario matrix of horizon prices — bootstrapped equities plus the cash column. One function builds it from a seed, so the optimizer's scenario set and every fresh set the evaluator grades on come out of the same generator.

In [5]:
def sample_market(seed: int, n_scenarios: int = N_SCENARIOS, history: np.ndarray = weekly_log_returns) -> np.ndarray:
    # joint projection of the invariants, invariant -> price, then the money-market column
    market_samples = scenario_market.sample_block_bootstrap_scenarios(
        weekly_log_returns=history,
        prices_vector=prices_vector,
        horizon_weeks=HORIZON_WEEKS,
        n_scenarios=n_scenarios,
        mean_block_length=mean_block_length,
        seed=seed,
    )
    return scenario_market.append_cash_column(market_samples, cash_horizon_value)


market_samples = sample_market(SEED)

# bin in numpy and plot bars: go.Histogram would embed all raw samples
# (4 x 400k floats, several MB) into the notebook JSON on every save.
fig = make_subplots(rows=3, cols=1, subplot_titles=[f"{name}: horizon price" for name in asset_symbol_list])
for column_index in range(len(asset_symbol_list)):
    counts, bin_edges = np.histogram(market_samples[:, column_index], bins=200)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    fig.add_trace(go.Bar(x=bin_centers, y=counts, marker_line_width=0), row=column_index + 1, col=1)
fig.update_layout(showlegend=False, height=700)
fig.show()

horizon_log_returns = np.log(market_samples[:, :3] / prices_vector.ravel())
print("1-year log return, mean:", horizon_log_returns.mean(axis=0).round(4))
print("1-year log return, std: ", horizon_log_returns.std(axis=0).round(4))
print(f"cash column constant: {np.ptp(market_samples[:, 3]) == 0}")

1-year log return, mean: [0.1197 0.092  0.0631]
1-year log return, std:  [0.1374 0.1317 0.1505]
cash column constant: True


## Entropy pooling: prior $p_0$ → posterior $p$

The output of this stage is a probability vector $p$ over the *unchanged* scenarios, not new scenarios (Meucci 2008, "Fully Flexible Views"). Views — on expectations, quantiles, rankings, probabilities of events — are first converted into **linear constraints on $p$** (the "converter" box: $A p = b$, $G p \le h$; `scenario_market.ViewConstraints`), and the posterior is the distribution closest to the prior in relative entropy that satisfies them:

$$p = \arg\min_{q}\; \sum_j q_j \ln\frac{q_j}{p_{0,j}} \quad\text{s.t.}\quad A q = b,\; G q \le h,\; \textstyle\sum_j q_j = 1.$$

**Today this stage is a documented pass-through:** with no views the minimum-relative-entropy posterior is the prior itself, so `entropy_pooling` returns $p_0$ (and raises if views are passed, rather than silently ignoring them). The prior is uniform over the projected scenarios, $p_{0,j} = 1/J$ — each bootstrap path is an equally likely draw. The point of wiring it now: every consumer downstream already takes $(X, p)$, so the solver plugs in without touching them.

Two kinds of linear constraints, kept apart: **view constraints** act on probabilities (here); **portfolio constraints** act on weights (in the optimizer, next).

In [6]:
def market_distribution(seed: int, n_scenarios: int = N_SCENARIOS, history: np.ndarray = weekly_log_returns):
    # (X, p): scenarios and their posterior probabilities — what the optimizer and the evaluator consume
    scenarios = sample_market(seed, n_scenarios, history)
    prior_probabilities = scenario_market.uniform_prior(len(scenarios))
    posterior_probabilities = scenario_market.entropy_pooling(scenarios, prior_probabilities, view_constraints=None)
    return scenarios, posterior_probabilities


probabilities = scenario_market.entropy_pooling(
    market_samples, scenario_market.uniform_prior(len(market_samples)), view_constraints=None
)
print(f"posterior == prior: {np.array_equal(probabilities, scenario_market.uniform_prior(len(market_samples)))}")

posterior == prior: True


## Portfolio optimization: max CE s.t. CVaR budget, probability-weighted (SAA)

The program of paper 1's step 4 with explicit probabilities — goal: absolute portfolio value at the horizon; utility: CARA; index of satisfaction: certainty-equivalent; risk: CVaR in the loss convention. In weights $h_i = p_i\alpha_i/w$ with scenario gross returns $g_j$ and relative risk tolerance $\zeta/w$:

$$\max_h\; -\tfrac{\zeta}{w}\,\ln\sum_j p_j\, e^{-g_j'h\,/(\zeta/w)} \quad\text{s.t.}\quad \min_t\Big\{t + \tfrac{1}{1-c}\sum_j p_j\,(1 - g_j'h - t)^+\Big\} \le \gamma,\quad \mathbf 1'h = 1,\quad h \ge 0.$$

The weights enter `cp.log_sum_exp(expr + log p)`; the Rockafellar–Uryasev CVaR is $t + p'(L - t)^+/(1-c)$. Uniform $p$ reproduces the unweighted SAA of paper 1 (checked to solver tolerance). The solver sees the first 10,000 scenarios (probabilities renormalized).

**Portfolio constraints** (on weights, separate from the view constraints on $p$): full investment and **long-only**, $h \ge 0$ — explicit, not a by-product of the formulation. Per-asset caps (`max_weights`) are available and used only in the illustrative liquidity case below.

**CVaR budget $\gamma$ — placeholder.** Paper 1's 4.1% was calibrated as a one-week budget on an all-equity book; at one year and with cash available it is a different statement. Placeholder: **$\gamma$ = 5% of wealth** — "in the worst 1-in-20 years, lose no more than 5% on average". It is a policy number, not an estimate; the cell below also solves with the budget switched off to show whether it binds.

In [7]:
wealth = 100e6
risk_tolerance = 10e6
confidence_level = 0.95
gamma_cvar = 0.05  # PLACEHOLDER policy budget for a 1-year horizon (see text)
N_OPTIMIZATION_SCENARIOS = 10_000

print("CVaR budget switched off (gamma = 100%; t* is then arbitrary — the constraint is slack):")
alpha_unconstrained = scenario_market.optimization_with_cvar_constrain(
    market_samples=market_samples,
    prices_vector=prices_with_cash,
    wealth=wealth,
    risk_tolerance=risk_tolerance,
    confidence_level=confidence_level,
    gamma=1.0,
    n_optimization_scenarios=N_OPTIMIZATION_SCENARIOS,
    probabilities=probabilities,
)
unconstrained_scores = scenario_market.score_allocation(
    alpha_unconstrained, market_samples, wealth, risk_tolerance, confidence_level, probabilities
)
print(f"its CVaR on the full set: {unconstrained_scores.cvar:,.0f}  vs budget {gamma_cvar * wealth:,.0f}"
      f"  -> budget binds: {unconstrained_scores.cvar > gamma_cvar * wealth}")

print(f"\nCVaR budget gamma = {gamma_cvar:.0%}:")
alpha_scenario = scenario_market.optimization_with_cvar_constrain(
    market_samples=market_samples,
    prices_vector=prices_with_cash,
    wealth=wealth,
    risk_tolerance=risk_tolerance,
    confidence_level=confidence_level,
    gamma=gamma_cvar,
    n_optimization_scenarios=N_OPTIMIZATION_SCENARIOS,
    probabilities=probabilities,
)
weights_scenario = alpha_scenario.ravel() * prices_with_cash / wealth
print("weights:", pd.Series(weights_scenario, index=asset_names).round(4).to_dict())

CVaR budget switched off (gamma = 100%; t* is then arbitrary — the constraint is slack):


status: optimal
Allocations: [  554733.   122798.        0. 50563746.]
CE (in-solver, optimistically biased): 105,544,325
VaR proxy t*: 35,633,744
its CVaR on the full set: 6,548,041  vs budget 5,000,000  -> budget binds: True

CVaR budget gamma = 5%:


status: optimal
Allocations: [  466719.    98178.        0. 58616771.]
CE (in-solver, optimistically biased): 105,479,614
VaR proxy t*: 2,601,886
weights: {'DFUS': 0.3738, 'DFAI': 0.04, 'DFAE': 0.0, 'Cash': 0.5862}


## Illiquidity model: Meucci (2012) parameters and funding policy

The illiquidity box of the diagram, shared by the next two stages: the liquidity-aware optimizer and the evaluator's liquidity grading (c). Selling at the horizon costs money and is uncertain. Meucci, "A Fully Integrated Liquidity and Market Risk Model" (*FAJ* 2012), overlays that on the scenario P&L: in scenario $j$ (probability $p_j$, market P&L $\bar\pi_j$) a **liquidation policy** decides what is sold, $\Delta h_{j}$; the **liquidity adjustment** $\Delta\Pi \mid j \sim N(\mu_j, \sigma_j^2)$ follows from the VWAP execution moments

$$\mu_n = -\alpha_n e_n|\Delta h_n| - \beta_n e_n \bar\sigma_n \frac{|\Delta h_n|^{3/2}}{\sqrt{v_n}} \;\;(6),\qquad \sigma_n = \delta_n\sqrt{v_n}\,e_n|\Delta h_n| \;\;(7),$$
$$\rho_{nm} = \gamma_\rho + (1-\gamma_\rho)\hat\rho_{nm} \;\;(8),\qquad \mu_j = \textstyle\sum_n \mu_n,\;\; \sigma_j^2 = \sum_{nm}\sigma_n\sigma_m\rho_{nm} \;\;(9),$$

— spread $\alpha$, square-root impact $\beta$, execution-period volatility $\bar\sigma$ and market volume $v$, uncertainty $\delta$, and correlations shrunk towards 1 by $\gamma_\rho$ = 0.9 (liquidity is less diversifiable than market risk; named $\gamma_\rho$ so it does not clash with the CVaR budget $\gamma$). $e_n$ is the horizon price in the scenario. $\delta_n = \bar\sigma_n/\sqrt{3 v_n}$ is not a free parameter: it is the paper's own VWAP result (A.2, eqs. 31 and 46) — the price risk of a position that shrinks linearly to zero.

**Funding need.** As in the paper's Example 4, cash must be raised where the 1-year market P&L falls below a threshold $\bar\pi$: the need is $(\bar\pi - \bar\pi_j)^+$. The threshold is a **dollar policy number** — placeholder USD −6M: a year that loses more than 6% of wealth forces sales of the excess — not a multiple of the chosen portfolio's P&L volatility, which would make the policy depend on the allocation it is meant to constrain.

**Liquidation policy: cheapest sales first** (`liquidity.cost_minimizing_liquidation_policy`). The need is met by the sales of least expected cost: cash first (costless), then the assets of least marginal cost — the paper's own rule (§2.3: a firm under funding pressure liquidates "those securities with the smallest marginal cost of liquidation"), and exactly the recourse the optimizer chooses. In dollars sold $S_n = e_n|\Delta h_n|$, (6) reads $-\mu_n = \alpha_n S_n + k_n S_n^{3/2}$ with $k_n = \beta_n\bar\sigma_n/\sqrt{e_n v_n}$ — square-root impact against the *dollar* volume of the execution period (`liquidity.sale_impact_coefficients`). Proceeds are **net**: spread and impact are paid out of the sale, so the need is covered by $\sum_n (S_n + \mu_n)$. Minimizing cost equalizes marginal costs, $\alpha_n + \tfrac32 k_n\sqrt{S_n} = \kappa$ (water-filling), solved per scenario by bisection on $\kappa$ — exact, no solver. The cost variance does not enter the choice; the conditional convolution measures it.


In [8]:
# ---- Liquidity parameters: PLACEHOLDERS, not data ---------------------------------------
# The PriceFetcher returns closes only (no volume, no quotes), so spreads and volumes below are
# order-of-magnitude ASSUMPTIONS for US-listed Dimensional ETFs, to be replaced by exchange data
# (consolidated volume, quoted spreads) before any conclusion is drawn.  For ETFs, exchange
# volume also understates the liquidity available through creation/redemption by APs.
# Only volatility and correlation come from the data (the weekly invariants).
EXECUTION_DAYS = 1  # tau_n: every sale executed over one trading day (the paper's case study)
liquidity_spread = np.array([2e-4, 3e-4, 5e-4, 0.0])  # alpha_n: half spread + commission, PLACEHOLDER
liquidity_impact = np.array([1.0, 1.0, 1.0, 0.0])  # beta_n: square-root law, Y ~ 1 (Toth et al. 2011), PLACEHOLDER
average_daily_volume_usd = np.array([40e6, 60e6, 30e6])  # USD per day, PLACEHOLDER
FUNDING_THRESHOLD_USD = -6e6  # funding need below this 1-year market P&L, dollars: PLACEHOLDER policy number

# derived: volume in units over the execution period, execution-period volatility from the
# weekly invariants (sqrt-time from 5 trading days), market correlations; cash is costless
market_correlation = np.eye(len(asset_names))
market_correlation[:3, :3] = np.corrcoef(weekly_log_returns, rowvar=False)


def build_liquidity_parameters(daily_volume_usd: np.ndarray) -> liquidity.LiquidityParameters:
    # one function, so that the illustrative volume stress below changes nothing else
    execution_volume = np.append(daily_volume_usd / prices_vector.ravel() * EXECUTION_DAYS, 1.0)
    execution_volatility = np.append(weekly_log_returns.std(axis=0, ddof=1) * np.sqrt(EXECUTION_DAYS / 5), 0.0)
    return liquidity.LiquidityParameters(
        spread=liquidity_spread,
        impact=liquidity_impact,
        volatility=execution_volatility,
        volume=execution_volume,
        uncertainty=liquidity.vwap_uncertainty_coefficient(execution_volatility, execution_volume),
        market_correlation=market_correlation,
        correlation_shrinkage=liquidity.LIQUIDITY_CORRELATION_SHRINKAGE,  # gamma_rho = 0.9
    )


liquidity_parameters = build_liquidity_parameters(average_daily_volume_usd)
execution_volatility = liquidity_parameters.volatility
print("plain allocation, positions vs volume:")
position_value = alpha_scenario.ravel() * prices_with_cash
print(pd.DataFrame({'position USD': position_value.round(), 'ADV USD': np.append(average_daily_volume_usd, np.nan),
                    'position / ADV': np.append(position_value[:3] / average_daily_volume_usd, np.nan).round(3),
                    'exec. vol': execution_volatility.round(4)}, index=asset_names))

plain allocation, positions vs volume:
      position USD     ADV USD  position / ADV  exec. vol
DFUS    37379540.0  40000000.0           0.934     0.0099
DFAI     4003689.0  60000000.0           0.067     0.0093
DFAE           0.0  30000000.0           0.000     0.0107
Cash    58616771.0         NaN             NaN     0.0000


## Liquidity-aware portfolio optimization

The optimizer of the previous section with the illiquidity model inside — a two-stage stochastic program: the weights $h$ are chosen today; in each scenario the sales $s_j \ge 0$ (fractions of wealth) are a *recourse* decision taken at the horizon. With gross returns $g_j$, relative risk tolerance $\zeta' = \zeta/w$, threshold $\bar\pi$ and the cost of (6) per unit of wealth, $c_{jn}(s) = \alpha_n s + k_{jn}\sqrt{w}\, s^{3/2}$ (the $\sqrt{w}$ turns dollars into fractions of wealth):

$$\max_{h,\,s,\,t}\; -\zeta'\ln\sum_j p_j \exp\Big(-\frac{g_j'h - \sum_n c_{jn}(s_{jn})}{\zeta'} + \frac{\lVert s_j M\rVert^2}{2\zeta'^2}\Big)$$

$$\text{s.t.}\quad t + \tfrac{1}{1-c}\textstyle\sum_j p_j\big(1 - g_j'h + \sum_n c_{jn}(s_{jn}) - t\big)^+ \le \gamma,\qquad \sum_n \big(s_{jn} - c_{jn}(s_{jn})\big) \ge \big(1 + \bar\pi/w - g_j'h\big)^+,$$

$$0 \le s_{jn} \le g_{jn}h_n,\qquad \mathbf 1'h = 1,\quad h \ge 0.$$

- **Objective**: the closed-form CE of the conditional convolution (evaluator (c)) on the solver scenarios: each scenario's wealth net of its expected cost $-\mu_j$, plus the CARA charge of its cost variance, $\sigma_j = w\lVert s_j M\rVert$ — eqs. (7)–(9) as a sum of squares (`liquidity.liquidity_std_factor`). DCP: the exponent is convex inside an increasing log-sum-exp.
- **CVaR budget** on the loss *net of the expected cost*. $\sigma_j$ is left out of it: the evaluator measures the CVaR of the full mixture exactly.
- **Funding need**: $(\cdot)^+$ is convex in $h$ (`cp.pos`), net proceeds are concave in $s$ — concave $\ge$ convex, still DCP. The need is a hard constraint; all cash is always feasible.
- **One model**: $k$ and $M$ come from `liquidity`, from the same `LiquidityParameters` and threshold as the evaluator. For fixed $h$ the solver picks, per scenario, the cheapest sales that meet the need — the evaluator's cost-minimizing policy (up to $\sigma_j$'s tiny weight in the CE, which the policy ignores). Checked below.
- **Exact pre-screen**: a scenario in which no asset falls below $1 + \bar\pi/w$ can never need funding, whatever $h$, and gets no sale variables.
- **Reductions** (checked below): zeroed liquidity costs, or a threshold no scenario reaches, reproduce the plain optimizer.

Same 10,000 solver scenarios and CLARABEL settings ladder as the plain optimizer.

In [9]:
liquidity_allocation = scenario_market.optimization_with_cvar_and_liquidity(
    market_samples=market_samples,
    prices_vector=prices_with_cash,
    wealth=wealth,
    risk_tolerance=risk_tolerance,
    confidence_level=confidence_level,
    gamma=gamma_cvar,
    funding_threshold=FUNDING_THRESHOLD_USD,
    liquidity_parameters=liquidity_parameters,
    n_optimization_scenarios=N_OPTIMIZATION_SCENARIOS,
    probabilities=probabilities,
)
alpha_liquidity = liquidity_allocation.allocation
weights_liquidity = alpha_liquidity.ravel() * prices_with_cash / wealth
print(pd.DataFrame({'plain': weights_scenario, 'liquidity-aware': weights_liquidity,
                    'difference': weights_liquidity - weights_scenario}, index=asset_names).round(6))

# reductions: no costs at all, or a need no scenario can reach -> the plain optimizer
costless_parameters = dataclasses.replace(
    liquidity_parameters,
    spread=np.zeros_like(liquidity_spread),
    impact=np.zeros_like(liquidity_impact),
    uncertainty=np.zeros_like(liquidity_parameters.uncertainty),
)
for label, threshold, parameters in [('zeroed liquidity costs', FUNDING_THRESHOLD_USD, costless_parameters),
                                     ('threshold never reached', -10 * wealth, liquidity_parameters)]:
    with contextlib.redirect_stdout(io.StringIO()):
        reduction = scenario_market.optimization_with_cvar_and_liquidity(
            market_samples, prices_with_cash, wealth, risk_tolerance, confidence_level, gamma_cvar,
            threshold, parameters, N_OPTIMIZATION_SCENARIOS, probabilities,
        )
    weight_gap = np.abs(reduction.allocation.ravel() * prices_with_cash / wealth - weights_scenario).max()
    print(f"{label}: status {reduction.status}, max |weight - plain weight| {weight_gap:.1e}")


status: optimal
Allocations: [  466719.    98178.        0. 58616763.]
CE (in-solver, optimistically biased): 105,479,613
VaR proxy t*: 2,601,680
funding need: possible in 2,481 of 10,000 solver scenarios, triggered with probability 1.31%; E[liquidity cost] 0
         plain  liquidity-aware  difference
DFUS  0.373795         0.373795        -0.0
DFAI  0.040037         0.040037         0.0
DFAE  0.000000         0.000000         0.0
Cash  0.586168         0.586168        -0.0


zeroed liquidity costs: status optimal, max |weight - plain weight| 9.2e-09


threshold never reached: status optimal, max |weight - plain weight| 3.5e-08


### Does it bind? Not with these placeholders — and why

With the placeholder parameters the liquidity-aware weights equal the plain ones (to solver tolerance). This is a result, not a bug: the need is triggered in about 1% of the solver scenarios and is always met from the ~59% cash at zero cost, so no equity is ever sold. The cash buffer the CVaR budget already imposes is the cheapest liquidity there is. A need can reach the equities only when the loss exceeds cash plus $|\bar\pi|$ — with worst 1-year equity drawdowns of ~30%, only books above ~3/4 equity — and even then the cost of a one-day VWAP sale is $\bar\sigma_{\text{day}}\sqrt{S/\text{ADV}} \approx 1\%\times\sqrt{S/\text{ADV}}$ of what is sold, small against 1-year return differences. (The execution period drops out of the square-root cost: $\bar\sigma \propto \sqrt{\tau_n}$, $v \propto \tau_n$.)

**Illustrative stress — not a calibrated case, not a recommendation.** To see the mechanism work, take away the cash buffer and scale the volumes down:
- money market capped at **10%** (a mandate; `max_weights`), CVaR budget **15%** (5% is infeasible with ≥ 90% equities),
- funding threshold **USD 0**: every losing year must be funded,
- ADV divided by 1, 10, 100, 1,000 (the last: ~USD 40k/day for DFUS — the order of magnitude of an illiquid sleeve, far outside the square-root law's calibrated range).

The plain optimizer under the same cap and budget is the reference.

In [10]:
ILLUSTRATIVE_CASH_CAP = 0.10  # ILLUSTRATIVE mandate, not a recommendation
ILLUSTRATIVE_GAMMA = 0.15  # ILLUSTRATIVE CVaR budget (5% is infeasible with the cash cap)
ILLUSTRATIVE_FUNDING_THRESHOLD_USD = 0.0  # ILLUSTRATIVE: every losing year must be funded
ILLUSTRATIVE_VOLUME_DIVISORS = [1, 10, 100, 1000]
illustrative_max_weights = np.array([1.0, 1.0, 1.0, ILLUSTRATIVE_CASH_CAP])

with contextlib.redirect_stdout(io.StringIO()):
    alpha_illustrative_plain = scenario_market.optimization_with_cvar_constrain(
        market_samples, prices_with_cash, wealth, risk_tolerance, confidence_level, ILLUSTRATIVE_GAMMA,
        N_OPTIMIZATION_SCENARIOS, probabilities, max_weights=illustrative_max_weights,
    )
illustrative_weights_dict = {'plain (no liquidity)': alpha_illustrative_plain.ravel() * prices_with_cash / wealth}
illustrative_runs_dict = {}  # divisor -> (parameters, LiquidityAwareAllocation)
for volume_divisor in ILLUSTRATIVE_VOLUME_DIVISORS:
    stressed_parameters = build_liquidity_parameters(average_daily_volume_usd / volume_divisor)
    solver_log = io.StringIO()
    with contextlib.redirect_stdout(solver_log):
        stressed_allocation = scenario_market.optimization_with_cvar_and_liquidity(
            market_samples, prices_with_cash, wealth, risk_tolerance, confidence_level, ILLUSTRATIVE_GAMMA,
            ILLUSTRATIVE_FUNDING_THRESHOLD_USD, stressed_parameters, N_OPTIMIZATION_SCENARIOS, probabilities,
            max_weights=illustrative_max_weights,
        )
    illustrative_runs_dict[volume_divisor] = (stressed_parameters, stressed_allocation)
    illustrative_weights_dict[f'liquidity-aware, ADV / {volume_divisor:,}'] = (
        stressed_allocation.allocation.ravel() * prices_with_cash / wealth
    )
    print(f"ADV / {volume_divisor:,}: " + solver_log.getvalue().splitlines()[-1])
print(pd.DataFrame(illustrative_weights_dict, index=asset_names).T.round(4))


ADV / 1: funding need: possible in 4,069 of 10,000 solver scenarios, triggered with probability 17.93%; E[liquidity cost] 570


ADV / 10: funding need: possible in 4,069 of 10,000 solver scenarios, triggered with probability 17.92%; E[liquidity cost] 1,715


ADV / 100: funding need: possible in 4,069 of 10,000 solver scenarios, triggered with probability 17.94%; E[liquidity cost] 5,496


ADV / 1,000: funding need: possible in 4,069 of 10,000 solver scenarios, triggered with probability 17.94%; E[liquidity cost] 19,229
                                DFUS    DFAI    DFAE  Cash
plain (no liquidity)          0.6259  0.2741  0.0000   0.1
liquidity-aware, ADV / 1      0.6256  0.2744  0.0000   0.1
liquidity-aware, ADV / 10     0.6248  0.2752  0.0000   0.1
liquidity-aware, ADV / 100    0.6221  0.2779  0.0000   0.1
liquidity-aware, ADV / 1,000  0.6118  0.2876  0.0006   0.1


### Consistency: the optimizer's liquidity cost is the evaluator's $\mu_j$

For the chosen weights, the evaluator's cost-minimizing policy is run on the 10,000 solver scenarios and its $\mu_j$ from eqs. (6)–(9) (`liquidity_adjustment_moments`, the evaluator's own code path) is compared with the expected cost inside the solver, scenario by scenario. Shown for the placeholder case (nothing costly is sold) and for the most stressed illustrative case, where equities are sold.

In [11]:
def compare_solver_and_evaluator_cost(label, allocation_result, parameters, threshold):
    solver_samples = market_samples[:N_OPTIMIZATION_SCENARIOS]
    solver_probabilities = probabilities[:N_OPTIMIZATION_SCENARIOS] / probabilities[:N_OPTIMIZATION_SCENARIOS].sum()
    policy = liquidity.cost_minimizing_liquidation_policy(threshold, parameters)
    distribution = liquidity.total_pnl_distribution(
        allocation_result.allocation, solver_samples, solver_probabilities, wealth, policy, parameters
    )
    evaluator_cost = -distribution.liquidity_mean
    cost_gap = allocation_result.solver_expected_cost - evaluator_cost
    std_gap = allocation_result.solver_cost_std - distribution.liquidity_std
    print(f"{label}\n  scenarios with costly sales (evaluator policy): {np.sum(evaluator_cost > 0):,} of {len(evaluator_cost):,}"
          f"\n  E[cost]  solver {solver_probabilities @ allocation_result.solver_expected_cost:,.2f}"
          f"   evaluator {solver_probabilities @ evaluator_cost:,.2f}   (largest scenario cost {evaluator_cost.max():,.0f})"
          f"\n  per scenario: max |cost gap| {np.abs(cost_gap).max():,.2f}   max |sigma gap| {np.abs(std_gap).max():,.2f}")


compare_solver_and_evaluator_cost('placeholder parameters', liquidity_allocation, liquidity_parameters, FUNDING_THRESHOLD_USD)
most_stressed_parameters, most_stressed_allocation = illustrative_runs_dict[ILLUSTRATIVE_VOLUME_DIVISORS[-1]]
compare_solver_and_evaluator_cost(f'illustrative, ADV / {ILLUSTRATIVE_VOLUME_DIVISORS[-1]:,}', most_stressed_allocation,
                                  most_stressed_parameters, ILLUSTRATIVE_FUNDING_THRESHOLD_USD)


placeholder parameters
  scenarios with costly sales (evaluator policy): 0 of 10,000
  E[cost]  solver 0.06   evaluator 0.00   (largest scenario cost -0)
  per scenario: max |cost gap| 1.00   max |sigma gap| 18.67
illustrative, ADV / 1,000
  scenarios with costly sales (evaluator policy): 375 of 10,000
  E[cost]  solver 19,229.37   evaluator 19,229.33   (largest scenario cost 4,404,244)
  per scenario: max |cost gap| 2.07   max |sigma gap| 8.94


# Evaluator

The optimizer's own numbers are claims made on the scenarios it was allowed to fit: in-sample CE and CVaR are optimistically biased (the optimizer's curse). The evaluator re-scores the *chosen* weights — those of the liquidity-aware optimizer (with the placeholder parameters they coincide with the plain ones to solver tolerance) — three ways, each answering a different question:

- **(a) unseen scenarios** — Monte Carlo/SAA error: the same market model, a fresh seed.
- **(b) history resampling** — estimation error: a different, equally plausible history.
- **(c) liquidity** — what selling at the horizon costs: Meucci's (2012) market-plus-liquidity P&L.

## (a) Probability-weighted CE, CVaR and value on unseen scenarios

Three lines: the 10,000 scenarios the solver saw (reproduces the in-solver CE — here because the liquidity-aware optimizer's in-solver liquidity cost is ~0 at the placeholders; these are market-only yardsticks, liquidity is graded in (c)), the full 400,000-scenario set, and a fresh-seed set built by the same generator and the same entropy-pooling stage. All probability-weighted with that set's $p$.

Caveat inherited from paper 1: every bootstrap set — in-sample or fresh seed — resamples the same ~270 historical weeks. The fresh seed measures sampling noise of the projection and the optimizer's overfitting to its 10,000 rows, **not** estimation error. That is (b).

In [12]:
evaluation = scenario_market.evaluate_allocation(
    allocations=alpha_liquidity,
    market_samples=market_samples,
    sample_scenarios=market_distribution,
    wealth=wealth,
    risk_tolerance=risk_tolerance,
    confidence_level=confidence_level,
    gamma=gamma_cvar,
    out_of_sample_seed=SEED + 1,
    probabilities=probabilities,
    n_optimization_scenarios=N_OPTIMIZATION_SCENARIOS,
)
unseen_samples, unseen_probabilities = market_distribution(SEED + 1)  # the judge set, reused below
print(f"\nin-solver minus unseen: CE {evaluation.in_solver.certainty_equivalent - evaluation.unseen.certainty_equivalent:+,.0f}"
      f"   CVaR {evaluation.in_solver.cvar - evaluation.unseen.cvar:+,.0f}")

in-solver    : CE 105,479,614   CVaR 4,999,998   E[value] 107,329,464   (budget 5,000,000)
full set     : CE 105,471,954   CVaR 5,001,880   E[value] 107,329,052   (budget 5,000,000)


out-of-sample: CE 105,466,470   CVaR 5,009,324   E[value] 107,328,297   (budget 5,000,000)



in-solver minus unseen: CE +13,144   CVaR -9,325


## (b) History resampling: estimation uncertainty

The whole input of the model is ~270 weeks of history; a different 5-year window would have given a different empirical distribution and a different allocation. To see how much, resample the history itself — a stationary block bootstrap of the historical weeks, same length, same block scheme (`scenario_market.resample_history`) — and push each replica through the full pipeline: projection → cash → entropy pooling → optimizer. $M$ = 50 replicas.

Two control choices:
- **A fixed judge.** Each replica's allocation is also scored on the unseen set of (a), built from the *observed* history: the CE *lost* to estimation error, measured on one common yardstick (paired comparison).
- **A Monte Carlo baseline.** The same pipeline on the *observed* history with fresh projection seeds only (20 runs): the spread that is pure SAA noise. The excess of (b) over it is estimation error.

**Plain optimizer here, not the liquidity-aware one.** Run through the liquidity-aware optimizer (placeholder parameters, threshold USD −6M), the same 70 pipelines took ~20 minutes instead of ~1 (median 5 s per solve, up to 2 minutes when the settings ladder runs out), 5 of the 50 history replicas failed on every rung — excluding them would bias the spread — and on every replica that solved, the weights matched the plain optimizer's to ≤ 5e-4 (median 1e-7): with these parameters cash funds every need for free, so liquidity does not move the allocation. Revisit once liquidity inputs are real.

Runtime: each replica projects 10,000 scenarios — exactly the rows its optimizer sees. Solver statuses are counted and reported: at one year CLARABEL needs a ladder of settings (see `scenario_market._CLARABEL_SETTINGS_LADDER`), and a replica that ends `optimal_inaccurate` is kept but flagged, one that fails is excluded and counted.

In [13]:
N_HISTORY_REPLICAS = 50
N_MC_BASELINE = 20
HISTORY_SEED_BASE = 10_000
PROJECTION_SEED_BASE = 20_000


def run_pipeline(history: np.ndarray, projection_seed: int) -> dict:
    scenarios, scenario_probabilities = market_distribution(projection_seed, N_OPTIMIZATION_SCENARIOS, history)
    solver_log = io.StringIO()
    try:
        with contextlib.redirect_stdout(solver_log):  # the optimizer prints; keep the loop quiet
            allocation = scenario_market.optimization_with_cvar_constrain(
                market_samples=scenarios,
                prices_vector=prices_with_cash,
                wealth=wealth,
                risk_tolerance=risk_tolerance,
                confidence_level=confidence_level,
                gamma=gamma_cvar,
                n_optimization_scenarios=N_OPTIMIZATION_SCENARIOS,
                probabilities=scenario_probabilities,
            )
    except RuntimeError:  # no settings rung found a solution: count it, do not hide it
        return {'status': 'failed'}
    in_sample = scenario_market.score_allocation(
        allocation, scenarios, wealth, risk_tolerance, confidence_level, scenario_probabilities
    )
    judged = scenario_market.score_allocation(
        allocation, unseen_samples, wealth, risk_tolerance, confidence_level, unseen_probabilities
    )
    return {
        'status': solver_log.getvalue().splitlines()[0].removeprefix('status: '),
        'weights': allocation.ravel() * prices_with_cash / wealth,
        'ce_own': in_sample.certainty_equivalent,  # CE the replica's optimizer reports to itself
        'ce_judged': judged.certainty_equivalent,  # CE of its weights under the observed-history judge
        'cvar_judged': judged.cvar,
    }


history_runs_all = [
    run_pipeline(scenario_market.resample_history(weekly_log_returns, mean_block_length, HISTORY_SEED_BASE + m),
                 PROJECTION_SEED_BASE + m)
    for m in range(N_HISTORY_REPLICAS)
]
mc_runs_all = [run_pipeline(weekly_log_returns, PROJECTION_SEED_BASE + m) for m in range(N_MC_BASELINE)]

for label, all_runs in [('history resampling', history_runs_all), ('MC baseline (observed history)', mc_runs_all)]:
    statuses = pd.Series([run['status'] for run in all_runs]).value_counts().to_dict()
    runs = [run for run in all_runs if run['status'] != 'failed']
    weights_matrix = np.array([run['weights'] for run in runs])
    print(f"\n{label}: {len(all_runs)} runs, solver status {statuses}")
    summary = pd.DataFrame(
        {'mean': weights_matrix.mean(axis=0), 'std': weights_matrix.std(axis=0, ddof=1),
         'p5': np.percentile(weights_matrix, 5, axis=0), 'p95': np.percentile(weights_matrix, 95, axis=0)},
        index=asset_names,
    )
    print(summary.round(3))
    for key, name in [('ce_own', 'CE, own scenarios'), ('ce_judged', 'CE, fixed judge'), ('cvar_judged', 'CVaR, fixed judge')]:
        values = np.array([run[key] for run in runs])
        print(f"{name:18s} mean {values.mean():,.0f}   std {values.std(ddof=1):,.0f}   "
              f"p5 {np.percentile(values, 5):,.0f}   p95 {np.percentile(values, 95):,.0f}")

history_runs = [run for run in history_runs_all if run['status'] != 'failed']
mc_runs = [run for run in mc_runs_all if run['status'] != 'failed']
judged_ce = np.array([run['ce_judged'] for run in history_runs])
print(f"\nCE lost to estimation error under the fixed judge (vs the observed-history optimum "
      f"{evaluation.unseen.certainty_equivalent:,.0f}): mean {evaluation.unseen.certainty_equivalent - judged_ce.mean():,.0f}")
print(f"replicas breaching the budget under the fixed judge: "
      f"{np.mean([run['cvar_judged'] > gamma_cvar * wealth for run in history_runs]):.0%}")

fig = go.Figure()
for column_index, name in enumerate(asset_names):
    fig.add_trace(go.Box(y=[run['weights'][column_index] for run in history_runs], name=f"{name} (history)"))
    fig.add_trace(go.Box(y=[run['weights'][column_index] for run in mc_runs], name=f"{name} (MC only)"))
fig.update_layout(title='Optimal weights: history resampling vs Monte Carlo noise alone', yaxis_title='weight', showlegend=False)
fig.show()

/Users/offbeat/Projects/emfin_capstone/emfin_capstone/toolbox/scenario_market.py:95: UserWarning:

Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.




history resampling: 50 runs, solver status {'optimal': 49, 'optimal_inaccurate': 1}
       mean    std   p5    p95
DFUS  0.290  0.285  0.0  0.860
DFAI  0.158  0.246  0.0  0.685
DFAE  0.021  0.056  0.0  0.152
Cash  0.531  0.309  0.0  1.000
CE, own scenarios  mean 107,002,085   std 3,974,560   p5 103,000,000   p95 114,912,118
CE, fixed judge    mean 104,228,825   std 1,107,310   p5 102,381,855   p95 105,505,256
CVaR, fixed judge  mean 6,223,867   std 6,007,631   p5 -3,000,000   p95 16,489,790

MC baseline (observed history): 20 runs, solver status {'optimal': 20}
       mean    std     p5    p95
DFUS  0.374  0.009  0.359  0.384
DFAI  0.038  0.011  0.022  0.056
DFAE  0.000  0.000  0.000  0.000
Cash  0.589  0.008  0.579  0.599
CE, own scenarios  mean 105,451,956   std 52,520   p5 105,355,579   p95 105,516,865
CE, fixed judge    mean 105,462,446   std 10,378   p5 105,449,660   p95 105,479,316
CVaR, fixed judge  mean 4,974,341   std 125,358   p5 4,815,979   p95 5,178,750

CE lost to estimat

## (c) Liquidity: Meucci (2012) market-plus-liquidity P&L

The liquidity moments (6)–(9), the parameters and the funding policy are those of the illiquidity section above — the same ones the optimizer used. Here they grade the chosen allocation exactly.

The total P&L $\Pi = \bar\Pi + \Delta\Pi$ is then a **mixture of normals** — the conditional convolution, eq. (10):

$$f_\Pi(y) = \sum_j \frac{p_j}{\sigma_j}\,\varphi\Big(\frac{y - \bar\pi_j - \mu_j}{\sigma_j}\Big),\qquad F_\Pi(y) = \sum_j p_j\,\Phi\Big(\frac{y - \bar\pi_j - \mu_j}{\sigma_j}\Big).$$

A scenario where nothing is sold has $\sigma_j = 0$: a point mass, handled exactly (step in the CDF, fractional boundary atom in the CVaR), never as $0/0$. Everything below is closed form in (10) — no grid, no quadrature: the quantile $z = F_\Pi^{-1}(\alpha)$ by Brent's method on the exact CDF, $\alpha = 1 - c$, and with $z_j = (z - \bar\pi_j - \mu_j)/\sigma_j$

$$\mathrm{CVaR}\{\Pi\} = \underbrace{\tfrac1\alpha\textstyle\sum_j p_j\,\bar\pi_j\,\Phi(z_j)}_{\text{market}} + \underbrace{\tfrac1\alpha\textstyle\sum_j p_j\big(\mu_j\Phi(z_j) - \sigma_j\varphi(z_j)\big)}_{\text{liquidity}},$$

$$\mathrm{LS} = \frac{\mathrm{CVaR}\{\bar\Pi\}}{\mathrm{CVaR}\{\Pi\}}\;(13),\qquad \mathrm{CE} = -\zeta\ln\sum_j p_j\exp\Big(-\frac{w + \bar\pi_j + \mu_j}{\zeta} + \frac{\sigma_j^2}{2\zeta^2}\Big).$$

CVaR here is the paper's left-tail mean $E[\Pi \mid \Pi \le z]$ (negative = loss); the liquidity score compares losses (1 = liquid, → 0 illiquid). **The liquidity term is the corrected one**: the printed eq. (12)/(58) keeps the unconditional weights $p_j$ where conditioning on the tail requires $p_j\Phi(z_j)/\alpha$ (see `notes/meucci2012_liquidity_errata.md`); it is verified against brute-force Monte Carlo at the end of this section.

**Policies** (swappable functions, `liquidity.LiquidationPolicy`), all with the same dollar threshold $\bar\pi$:
- *Cost-minimizing funding* — **the default, the optimizer's own policy**: where the market P&L falls below $\bar\pi$, raise $(\bar\pi - \bar\pi_j)$ net of costs by the cheapest sales (cash first).
- *Proportional funding* (paper Example 4) — stress: raise $\bar\pi - \bar\pi_j$ (gross proceeds) by selling the same fraction $\theta_j$ of every holding (cash included); elsewhere sell nothing.
- *Full liquidation* (paper Example 2) — stress: sell everything at the horizon in every scenario.

In [14]:
tail_probability = 1 - confidence_level  # alpha


def policies_for(threshold, parameters):
    # the shared policy first (default), then the stress policies; one dollar threshold for all
    return {
        f"cost-minimizing funding below {threshold:,.0f} (shared with the optimizer)":
            liquidity.cost_minimizing_liquidation_policy(threshold, parameters),
        f"proportional funding below {threshold:,.0f} (Example 4, stress)": liquidity.funding_liquidation_policy(threshold),
        "full liquidation (Example 2, stress)": liquidity.full_liquidation_policy,
    }


policies_dict = policies_for(FUNDING_THRESHOLD_USD, liquidity_parameters)
distributions_dict = {}
for policy_name, liquidation_policy in policies_dict.items():
    distribution = liquidity.total_pnl_distribution(
        holdings=alpha_liquidity,
        horizon_prices=unseen_samples,
        probabilities=unseen_probabilities,
        wealth=wealth,
        liquidation_policy=liquidation_policy,
        parameters=liquidity_parameters,
    )
    distributions_dict[policy_name] = distribution
    split = liquidity.total_pnl_cvar_split(distribution, tail_probability)
    market_split = liquidity.total_pnl_cvar_split(liquidity.market_only(distribution), tail_probability)
    print(f"\n{policy_name}")
    print(f"  sells in {unseen_probabilities @ (distribution.liquidity_std > 0):.1%} of scenarios;"
          f"  E[liquidity adjustment] {unseen_probabilities @ distribution.liquidity_mean:,.0f}")
    print(f"  CVaR (left tail, P&L)  market-only {market_split.cvar:,.0f}   total {split.cvar:,.0f}")
    print(f"  split of total CVaR    market {split.market_contribution:,.0f}   liquidity {split.liquidity_contribution:,.0f}")
    print(f"  VaR quantile z         market-only {market_split.quantile:,.0f}   total {split.quantile:,.0f}")
    print(f"  liquidity score LS     {liquidity.liquidity_score(distribution, tail_probability):.5f}")
    print(f"  CE                     market-only {liquidity.total_pnl_certainty_equivalent(liquidity.market_only(distribution), wealth, risk_tolerance):,.0f}"
          f"   with liquidity {liquidity.total_pnl_certainty_equivalent(distribution, wealth, risk_tolerance):,.0f}")

# plot through CDF differences: the funding policy leaves most scenarios as point masses,
# which a density cannot show; bin probabilities can
fig = make_subplots(rows=1, cols=3, subplot_titles=list(distributions_dict))
for column_index, distribution in enumerate(distributions_dict.values()):
    bin_edges = np.linspace(np.percentile(distribution.market_pnl, 0.2) - 2e6, np.percentile(distribution.market_pnl, 99.8), 121)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    market_mass = np.diff(liquidity.total_pnl_cdf(liquidity.market_only(distribution), bin_edges))
    total_mass = np.diff(liquidity.total_pnl_cdf(distribution, bin_edges))
    fig.add_trace(go.Bar(x=bin_centers, y=market_mass, name='pure market P&L', marker_color='grey',
                         marker_line_width=0, showlegend=column_index == 0), row=1, col=column_index + 1)
    fig.add_trace(go.Scatter(x=bin_centers, y=total_mass, name='market + liquidity P&L', line_color='red',
                             showlegend=column_index == 0), row=1, col=column_index + 1)
fig.update_layout(title='1-year P&L of the chosen allocation, unseen scenarios (probability per bin)', bargap=0)
fig.show()


cost-minimizing funding below -6,000,000 (shared with the optimizer)
  sells in 0.0% of scenarios;  E[liquidity adjustment] 0
  CVaR (left tail, P&L)  market-only -5,009,324   total -5,009,324
  split of total CVaR    market -5,009,324   liquidity 0
  VaR quantile z         market-only -2,738,711   total -2,738,711


  liquidity score LS     1.00000
  CE                     market-only 105,466,470   with liquidity 105,466,470



proportional funding below -6,000,000 (Example 4, stress)
  sells in 1.2% of scenarios;  E[liquidity adjustment] -13
  CVaR (left tail, P&L)  market-only -5,009,324   total -5,009,583
  split of total CVaR    market -5,009,324   liquidity -260
  VaR quantile z         market-only -2,738,711   total -2,738,711


  liquidity score LS     0.99995
  CE                     market-only 105,466,470   with liquidity 105,466,410



full liquidation (Example 2, stress)
  sells in 100.0% of scenarios;  E[liquidity adjustment] -428,424
  CVaR (left tail, P&L)  market-only -5,009,324   total -5,331,200
  split of total CVaR    market -5,001,336   liquidity -329,864
  VaR quantile z         market-only -2,738,711   total -3,080,152


  liquidity score LS     0.93962
  CE                     market-only 105,466,470   with liquidity 105,069,036


### Liquidity-aware vs plain allocation, graded by the evaluator

Both allocations on the unseen scenarios of (a), under the shared policy and the two stress policies: CE and CVaR of the total P&L (market + liquidity, eq. 10), against the market-only CE. With the placeholders the two allocations are the same, so the grades are too. In the most stressed illustrative case the liquidity-aware allocation should give up a little market-only CE to buy back more in liquidity — that is the trade it was built to make, now judged on scenarios it did not see and with the cost uncertainty it only partly saw (none in its CVaR).

In [15]:
def grade_allocations(allocations_dict, parameters, threshold, budget):
    grades_dict = {}
    for policy_name, policy in policies_for(threshold, parameters).items():
        for allocation_name, allocation in allocations_dict.items():
            distribution = liquidity.total_pnl_distribution(
                allocation, unseen_samples, unseen_probabilities, wealth, policy, parameters
            )
            split = liquidity.total_pnl_cvar_split(distribution, tail_probability)
            grades_dict[(policy_name.split(' (')[0].split(' below')[0], allocation_name)] = {
                'CE market-only': liquidity.total_pnl_certainty_equivalent(liquidity.market_only(distribution), wealth, risk_tolerance),
                'CE with liquidity': liquidity.total_pnl_certainty_equivalent(distribution, wealth, risk_tolerance),
                'CVaR loss with liquidity': -split.cvar,
                'liquidity part of CVaR': -split.liquidity_contribution,
                'E[liquidity cost]': -(unseen_probabilities @ distribution.liquidity_mean),
            }
    grades = pd.DataFrame(grades_dict).T
    print(f"CVaR budget {budget * wealth:,.0f}")
    print(grades.round(0).to_string(float_format=lambda value: f"{value:,.0f}"))
    return grades


print("Placeholder parameters (gamma = 5%, threshold USD -6M):")
placeholder_grades = grade_allocations({'plain': alpha_scenario, 'liquidity-aware': alpha_liquidity},
                                       liquidity_parameters, FUNDING_THRESHOLD_USD, gamma_cvar)
print(f"\nIllustrative stress (cash <= {ILLUSTRATIVE_CASH_CAP:.0%}, gamma = {ILLUSTRATIVE_GAMMA:.0%}, threshold USD 0, "
      f"ADV / {ILLUSTRATIVE_VOLUME_DIVISORS[-1]:,}):")
stressed_grades = grade_allocations({'plain': alpha_illustrative_plain, 'liquidity-aware': most_stressed_allocation.allocation},
                                    most_stressed_parameters, ILLUSTRATIVE_FUNDING_THRESHOLD_USD, ILLUSTRATIVE_GAMMA)
shared = stressed_grades.loc['cost-minimizing funding']
print(f"\nliquidity-aware minus plain, shared policy: CE with liquidity "
      f"{shared.loc['liquidity-aware', 'CE with liquidity'] - shared.loc['plain', 'CE with liquidity']:+,.0f}, "
      f"CVaR loss {shared.loc['liquidity-aware', 'CVaR loss with liquidity'] - shared.loc['plain', 'CVaR loss with liquidity']:+,.0f}, "
      f"CE market-only {shared.loc['liquidity-aware', 'CE market-only'] - shared.loc['plain', 'CE market-only']:+,.0f}")


Placeholder parameters (gamma = 5%, threshold USD -6M):


CVaR budget 5,000,000
                                         CE market-only  CE with liquidity  CVaR loss with liquidity  liquidity part of CVaR  E[liquidity cost]
cost-minimizing funding plain               105,466,470        105,466,470                 5,009,323                      -0                 -0
                        liquidity-aware     105,466,470        105,466,470                 5,009,324                      -0                 -0
proportional funding    plain               105,466,470        105,466,410                 5,009,582                     260                 13
                        liquidity-aware     105,466,470        105,466,410                 5,009,583                     260                 13
full liquidation        plain               105,466,470        105,069,036                 5,331,199                 329,864            428,424
                        liquidity-aware     105,466,470        105,069,036                 5,331,200              

CVaR budget 15,000,000
                                         CE market-only  CE with liquidity  CVaR loss with liquidity  liquidity part of CVaR  E[liquidity cost]
cost-minimizing funding plain               104,077,290        103,794,311                14,188,932                 397,082             19,854
                        liquidity-aware     104,075,248        103,797,549                14,158,960                 390,764             19,538
proportional funding    plain               104,077,290        103,199,778                15,474,158               1,682,451            118,558
                        liquidity-aware     104,075,248        103,216,121                15,423,218               1,655,155            116,651
full liquidation        plain               104,077,290         74,406,082                39,088,544              25,359,373         33,986,946
                        liquidity-aware     104,075,248         74,816,946                38,728,127             

### Verification of the CVaR split (and of the printed eq. 58)

Brute force, no formula: replicate every scenario $K$ times (stratified, so the scenario weights are exact), draw $\Delta\Pi \mid j \sim N(\mu_j, \sigma_j^2)$, sort the total P&L, take the $\alpha$-tail (fractional boundary draw), and average $\Delta\Pi$ over it. Compared with the corrected closed form and with the printed formulas of the paper,

$$\text{(58)}\;\; \sum_j p_j\Big(\mu_j - \sigma_j\frac{\varphi(z_j)}{\Phi(z_j)}\Big), \qquad \text{(12)}\;\; \sum_j p_j\frac{\mu_j - \sigma_j\varphi(z_j)}{\Phi(z_j)},$$

evaluated stably ($\varphi/\Phi$ through `erfcx`), so any failure is the formula's, not floating point's.

In [16]:
from scipy import special


def printed_eq58(distribution, quantile):
    # sum_j p_j (mu_j - sigma_j phi(z_j)/Phi(z_j)); point masses contribute mu_j
    smooth = distribution.liquidity_std > 0
    standardized = (quantile - distribution.market_pnl[smooth] - distribution.liquidity_mean[smooth]) / distribution.liquidity_std[smooth]
    inverse_mills = np.sqrt(2 / np.pi) / special.erfcx(-standardized / np.sqrt(2))  # phi/Phi, stable
    terms = distribution.liquidity_mean.copy()
    terms[smooth] -= distribution.liquidity_std[smooth] * inverse_mills
    return distribution.probabilities @ terms


def printed_eq12(distribution, quantile):
    # sum_j p_j (mu_j - sigma_j phi(z_j)) / Phi(z_j)
    smooth = distribution.liquidity_std > 0
    standardized = (quantile - distribution.market_pnl[smooth] - distribution.liquidity_mean[smooth]) / distribution.liquidity_std[smooth]
    with np.errstate(divide='ignore', over='ignore'):
        terms = distribution.liquidity_mean.copy()
        terms[smooth] = (distribution.liquidity_mean[smooth] / special.ndtr(standardized)
                         - distribution.liquidity_std[smooth] * np.sqrt(2 / np.pi) / special.erfcx(-standardized / np.sqrt(2)))
        return distribution.probabilities @ terms


def brute_force_liquidity_contribution(distribution, draws_per_scenario, seed):
    rng = np.random.default_rng(seed)
    scenario_index = np.repeat(np.arange(len(distribution.probabilities)), draws_per_scenario)
    draw_probabilities = np.repeat(distribution.probabilities, draws_per_scenario) / draws_per_scenario
    liquidity_draws = (distribution.liquidity_mean[scenario_index]
                       + distribution.liquidity_std[scenario_index] * rng.standard_normal(len(scenario_index)))
    total_pnl = distribution.market_pnl[scenario_index] + liquidity_draws
    worst_first = np.argsort(total_pnl)
    mass_before = np.cumsum(draw_probabilities[worst_first]) - draw_probabilities[worst_first]
    tail_weights = np.clip(tail_probability - mass_before, 0, draw_probabilities[worst_first])
    return tail_weights @ liquidity_draws[worst_first] / tail_probability


DRAWS_PER_SCENARIO = 20
for policy_name, distribution in distributions_dict.items():
    split = liquidity.total_pnl_cvar_split(distribution, tail_probability)
    brute_force = [brute_force_liquidity_contribution(distribution, DRAWS_PER_SCENARIO, seed) for seed in (1, 2, 3)]
    print(f"{policy_name}\n  liquidity contribution to CVaR:  corrected {split.liquidity_contribution:,.0f}   "
          f"brute-force MC {np.mean(brute_force):,.0f} (seeds {', '.join(f'{value:,.0f}' for value in brute_force)})   "
          f"printed (58) {printed_eq58(distribution, split.quantile):,.0f}   printed (12) {printed_eq12(distribution, split.quantile):,.0f}")

cost-minimizing funding below -6,000,000 (shared with the optimizer)
  liquidity contribution to CVaR:  corrected 0   brute-force MC 0 (seeds 0, 0, 0)   printed (58) 0   printed (12) 0


proportional funding below -6,000,000 (Example 4, stress)
  liquidity contribution to CVaR:  corrected -260   brute-force MC -260 (seeds -258, -263, -259)   printed (58) -13   printed (12) -13


full liquidation (Example 2, stress)
  liquidity contribution to CVaR:  corrected -329,864   brute-force MC -329,847 (seeds -329,672, -330,108, -329,760)   printed (58) -10,530,819   printed (12) -inf


## Open items

- **Entropy pooling solver and views.** The stage is a pass-through; the next step is the dual solver (one convex program in the number of views) and the converter from views to `ViewConstraints`.
- **Liquidity inputs are placeholders.** Spreads, ADV and the funding threshold in the parameter cell are assumptions, not data; the price fetcher exposes no volume. With them the liquidity effect is nil for this book — the liquidity-aware optimizer returns the plain weights, because the cash buffer funds every need — and it takes an illustrative stress (cash cap, every losing year funded, ADV / 100–1,000) to move the weights. The conclusion cannot be stronger than the inputs; the model will matter for genuinely illiquid sleeves (real estate, once its invariant exists).
- **Block bootstrap vs filtered historical simulation** — see the note in the projection section.
- **Money market as a constant** — see *Known limitations*.
- From the whiteboard: **transaction costs**, **rebalancing** (the one-period model assumes buy-and-hold over the whole year), **timeline**.